In [1]:
!pip install -q transformers accelerate bitsandbytes torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.5 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [3]:
import json

def analyze_ticket(ticket_text):

    prompt = f"""
You are an ERP support ticket triaging assistant.

Analyze the ticket and return ONLY valid JSON.

JSON format:
{{
  "category": "Finance | Inventory | HR | Procurement | IT",
  "erp_platform": "SAP | Oracle Fusion | Microsoft Dynamics | Unknown",
  "issue_type": "Issue | Change Request | Support Request | Information Request",
  "priority": "High | Medium | Low"
}}

Ticket:
{ticket_text}

JSON:
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    response_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return response_text

In [4]:
def route_ticket(parsed_data):
    category = parsed_data["category"]
    platform = parsed_data["erp_platform"]

    if category == "Finance" and platform == "SAP":
        return "SAP Finance Support Team"
    elif category == "HR":
        return "HR Systems Support Team"
    elif category == "Inventory":
        return "Inventory Operations Team"
    else:
        return "General ERP Support Team"

In [30]:
import re
import json

ticket = input("Enter ERP Support Ticket:\n")

raw_output = analyze_ticket(ticket)

print("\n--- LLM RAW OUTPUT ---")
print(raw_output)

try:
    json_start = raw_output.rfind("{")
    json_end = raw_output.rfind("}") + 1

    if json_start != -1 and json_end != -1:
        json_str = raw_output[json_start:json_end]
        parsed = json.loads(json_str)

        print("\n--- STRUCTURED OUTPUT ---")
        print(json.dumps(parsed, indent=2))

        assigned_team = route_ticket(parsed)

        print("\n--- ROUTING RESULT ---")
        print("Assigned To:", assigned_team)

    else:
        print("⚠ No valid JSON found.")

except Exception as e:
    print("⚠ JSON parsing failed:", e)

Enter ERP Support Ticket:
Payroll processing failed for this month.

--- LLM RAW OUTPUT ---

You are an ERP support ticket triaging assistant.

Analyze the ticket and return ONLY valid JSON.

JSON format:
{
  "category": "Finance | Inventory | HR | Procurement | IT",
  "erp_platform": "SAP | Oracle Fusion | Microsoft Dynamics | Unknown",
  "issue_type": "Issue | Change Request | Support Request | Information Request",
  "priority": "High | Medium | Low"
}

Ticket:
Payroll processing failed for this month.

JSON:
{
  "category": "HR",
  "erp_platform": "Unknown",
  "issue_type": "Issue",
  "priority": "High"
}

--- STRUCTURED OUTPUT ---
{
  "category": "HR",
  "erp_platform": "Unknown",
  "issue_type": "Issue",
  "priority": "High"
}

--- ROUTING RESULT ---
Assigned To: HR Systems Support Team
